# 01.1 — Data collection

Mục tiêu: kiểm tra RAW snapshot, schema và các nguồn trước khi áp dụng rule xử lý. Notebook chỉ đọc dữ liệu. Câu hỏi: nguồn có đủ thông tin để truy vết trước khi tạo release không? Trả lời dựa trên source, revision và checksum trong bảng dưới.

In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'configs').is_dir(): ROOT = ROOT.parent
config = json.loads((ROOT / 'configs/data/phase01_it_en_vi.json').read_text(encoding='utf-8'))

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()


In [2]:
rows = []
for source, spec in config['sources'].items():
    path = ROOT / 'data/raw' / source / f'{source}_raw.parquet'
    raw = pd.read_parquet(path)
    rows.append({'source': source, 'rows': len(raw), 'columns': ', '.join(raw.columns), 'source_revision': spec['source_revision'], 'sha256': sha256(path)})
sources = pd.DataFrame(rows)
sources

,source,rows,columns,source_revision,sha256
0,envitech_reasoning,15115,"en, vi, category",raw-snapshot-only,ba354d36e9eeb2b1afcefda9ea98f63f7bf5e315e80743...
1,tech_viet_translation,100767,"instruction, output, input",raw-snapshot-only,9be76ed6ba1643629211e3c8b60f5ef957a0c5b8ebd84e...
2,gnome,149,"en, vi",v1,f6e1d4826b0ccdd4ad1906a9cad551a16b816b52029cb1...
3,ubuntu,5056,"en, vi",v14.10,3840fc0b455fc2c3ac2b407a8ad991dae2f1fd443f7429...
4,kde4,42782,"en, vi",v2,0171299fe73223aea450e7667d388940b76da3013c6e4c...


In [3]:
source = 'tech_viet_translation'  # đổi tên source để xem nguồn khác
spec = config['sources'][source]
raw = pd.read_parquet(ROOT / 'data/raw' / source / f'{source}_raw.parquet')
raw[[spec['en'], spec['vi']]].head(10)

,instruction,output
0,London pop-up spy room pays homage to 'Goldfin...,London có một phòng trinh sát tạm thời (pop-up...
1,'Russia’s Google’ exits the country — Yandex p...,Nga vừa tuyên bố sẽ rút khỏi Russia’s Google v...
2,Intel sneaks out nine 14th-Gen CPUs without E-...,Intel đã bén duyên với chín bộ vi xử lý 14th-G...
3,Nvidia could lose up to $12 billion in revenue...,NVIDIA có thể mất tới 12 tỷ USD doanh thu nếu ...
4,Suunto’s new headphones finally made me apprec...,Suunto đã tạo ra tai nghe mới cuối cùng khiến ...
5,Apple Card promo offers boosted 6% cash back a...,Apple Card ưu đãi hiện tại cung cấp 6% tiền mặ...
6,AMD Ryzen 9 9950X vs Intel Core i9-14900KS: Ci...,AMD Ryzen 9 9950X vs Intel Core i9-14900KS: So...
7,Microsoft to Deliver Faster and Smaller Update...,Microsoft sẽ cung cấp các bản cập nhật nhanh h...
8,Review: Phanteks Glacier One 360 T30 Gen2 - 36...,Đánh giá: Phanteks Glacier One 360 T30 Gen2 - ...
9,Team Group PD20 Mini External SSD: High-Speed ...,Team Group PD20 Mini External SSD: Giải pháp l...


Kết luận cần nêu khi bảo vệ: dữ liệu được giữ ở `data/raw`, source revision/checksum được lưu để truy vết; notebook không tải lại hoặc ghi đè RAW.